# Data Preprocessing and Postprocessing for VASP Calculations

In [1]:
import sys
sys.path.append('../scripts')
from step1_prepare_opt_inputs import VASPInputPreparer
from step3_prepare_piezo_inputs import PiezoInputPreparer
from step2_opt_output_check import OptOutputCheck
from step4_piezo_output_check import PiezoOutputCheck
import numpy as np
import yaml
from pathlib import Path
NOTEBOOK_DIR = Path().resolve()
print (NOTEBOOK_DIR)
ROOT = NOTEBOOK_DIR.parent  # go up one level

print("ROOT:", ROOT)
config = yaml.safe_load(open(ROOT / "config.yaml"))

calcdir = ROOT / config["calcdir"] #your full path to where you want vasp calculations
potcar_path = ROOT / config["potcar_path"] #your full path to potcarfiles
qmofbase= ROOT/config["qmofbase"] #Path to where QMOF output files, structural and INCAR files are located for each MOF
resultstestdir= ROOT/config["resultstestdir"] #your main path for results files
moflists= ROOT/config["alldatafiles"]/"moflists.txt"
potcar_suffixes={'Na':'_pv','K': '_sv','Ca': '_sv','Sc': '_sv','Ti': '_sv','V': '_sv','Cr': '_pv','Mn': '_pv','Ga': '_d','Ge': '_d',
'Rb': '_sv','Sr': '_sv','Y': '_sv','Zr': '_sv','Nb': '_sv','Mo': '_sv','Tc': '_pv','Ru': '_pv','Rh': '_pv','In': '_d','Sn': '_d',
'Cs': '_sv','Ba': '_sv','Pr': '_3','Nd': '_3','Pm': '_3','Sm': '_3','Eu': '_3','Gd': '_3','Tb': '_3','Dy': '_3','Ho': '_3',
'Er': '_3','Tm': '_3','Yb': '_3','Lu': '_3','Hf': '_pv','Ta': '_pv','W': '_sv','Tl': '_d','Pb': '_d','Bi': '_d','Po': '_d',
'At': '_d','Fr': '_sv','Ra': '_sv','Eu':'_3','Yb':'_3','W':'_sv'}

# Prepare base folders
def run_optimisation(calcdir, potcar_path, density,potcar_suffixes, qmofbase, moflists):
    print ("Optimisation inputs creating")
    preparer = VASPInputPreparer(calcdir, potcar_path, density,potcar_suffixes, qmofbase, moflists)
    # Step 1: Create directories for each MOF
    preparer.create_mof_dirs()

    # Step 2: Generate POSCAR files with symmetry
    preparer.convert_contcar_to_poscar_sym()

    # Step 3: Prepare POTCAR, KPOINTS, INCAR
    preparer.prepare_all()

#Check optimisation calculations, and output failed calculation folders
def check_optimisation(calcdir,resultstestdir):
    print ("Optimisation outputs checking")
    checker = OptOutputCheck(calcdir,resultstestdir)
    failed_optfinal = checker.run_full_check()
    return failed_optfinal

#Prepare piezoelectric calculation folders after optimisation finishes
def run_piezoelectric(calcdir, failed_optfinal):
    print ("Piezo inputs creating")
    # Then prepare piezo inputs
    with open(failed_optfinal, "r") as f:
        failedopt = [line.strip() for line in f]
    #failed = ['CSD1234']  # list of optimisation failed MOFs
    piezo = PiezoInputPreparer(calcdir, failedopt)
    piezo.prepare_all_piezo_inputs()

""" Check the status of piezoelectric calculations (failed or finished without errors) and
write the results of piezoelectric tensors into CSV files."""
def check_piezoelectriccalcs(resultstestdir,calcdir,failed_optfinal,moflists):
    print ("Piezo outputs checking")
    checker_piezo=PiezoOutputCheck(resultstestdir,calcdir,failed_optfinal,mofrefcodes=np.loadtxt(moflists, dtype=str))
    failed_piezo = checker_piezo.checkpiezo_outcar()
    checker_piezo.extractpiezo_data()


/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/notebooks
ROOT: /Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening


In [14]:
run_optimisation(calcdir, potcar_path, density,potcar_suffixes, qmofbase, moflists)

Optimisation inputs creating
Creating MOF directories...
Directory exists: LIQBAP_FSR
Directory exists: GAXQAZ_FSR
Directory exists: PACGAE_FSR
Directory exists: HOPDIB_FSR
Directory exists: TAXHUX_FSR
['LIQBAP_FSR' 'GAXQAZ_FSR' 'PACGAE_FSR' 'HOPDIB_FSR' 'TAXHUX_FSR']
Converting CONTCAR → POSCAR using symmetry...
Entering: LIQBAP_FSR


/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/notebooks/../scripts/step1_prepare_opt_inputs.py:96: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structure = parser.get_structures()[0]
/Users/srinidhi/miniconda3/envs/becwork/lib/python3.12/site-packages/pymatgen/io/cif.py:1348: UserWarning: Issues encountered while parsing CIF: Skipping relative stoichiometry check because CIF does not contain formula keys.
  return self.parse_structures(*args, **kwargs)


Processed LIQBAP_FSR → sgnum 5
Entering: GAXQAZ_FSR
Processed GAXQAZ_FSR → sgnum 9
Entering: PACGAE_FSR
Processed PACGAE_FSR → sgnum 9
Entering: HOPDIB_FSR
Processed HOPDIB_FSR → sgnum 143
Entering: TAXHUX_FSR
Processed TAXHUX_FSR → sgnum 15
Preparing VASP input files (POTCAR, KPOINTS, INCAR)
Processing: LIQBAP_FSR
Processing: HOPDIB_FSR
Processing: PACGAE_FSR
Processing: TAXHUX_FSR
Processing: GAXQAZ_FSR


/Users/srinidhi/miniconda3/envs/becwork/lib/python3.12/site-packages/pymatgen/io/cif.py:1028: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  self.symmetry_operations = self.get_symops(data)  # type:ignore[assignment]
/Users/srinidhi/miniconda3/envs/becwork/lib/python3.12/site-packages/pymatgen/io/cif.py:1348: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
Skipping relative stoichiometry check because CIF does not contain formula keys.
  return self.parse_structures(*args, **kwargs)


In [2]:
failed_optfinal= check_optimisation(calcdir,resultstestdir)
print (failed_optfinal)

Optimisation outputs checking
LIQBAP_FSR /Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/LIQBAP_FSR
HOPDIB_FSR /Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/HOPDIB_FSR
Optimisation did not reach a convergence
PACGAE_FSR /Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/PACGAE_FSR
TAXHUX_FSR /Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/TAXHUX_FSR
GAXQAZ_FSR /Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/GAXQAZ_FSR
check1, Opt failed in OUTCAR check: ['HOPDIB_FSR']
check2, Opt failed due to NSW in OSZICAR check: []
Final failed list (do not continue): ['HOPDIB_FSR']
['HOPDIB_FSR']


In [1]:
#failed_optfinal ='/your/path/to/list/of/mofs/which/failed/in/optimisation/step'
run_piezoelectric(calcdir, failed_optfinal)

NameError: name 'run_piezoelectric' is not defined

In [3]:
check_piezoelectriccalcs(resultstestdir,calcdir,failed_optfinal,moflists)

Piezo outputs checking
Piezoelectric calc dint work ['HOPDIB_FSR']
/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/LIQBAP_FSR
LIQBAP_FSR already in clamped.csv, skipping
LIQBAP_FSR already in dynamic.csv, skipping
LIQBAP_FSR already in totale.csv, skipping
/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/.DS_Store
/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/HOPDIB_FSR
Piezo didn't finish properly: HOPDIB_FSR
/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/PACGAE_FSR
PACGAE_FSR already in clamped.csv, skipping
PACGAE_FSR already in dynamic.csv, skipping
PACGAE_FSR already in totale.csv, skipping
/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/data/mof_calcs/TAXHUX_FSR
TAXHUX_FSR already in clamped.csv, skipping
TAXHUX_FSR already in dynamic.csv, skipping
TAXHUX_FSR already in totale

In [4]:
""" Code to filter results of Piezoelectric constant 'e' obtained from highthroughput calculations.
A) Filters on magnetic moment 1) being equal with QMOF database and optimisation, 2) being equal in Optimisation and piezoelectric steps
B) Filters on Clamped and dynamic 'e' values wrt point group, i.e., 1) Check on some terms in 'e' should be zero as per symmetry
2) Check on some terms in 'e' should be nearly equal with a tolerance specified.
"""
from piezofilters_pointgroup import *
import numpy as np
import pandas as pd

##Change the input results files and output result files here based on the file you want to use##
qmofv13= ROOT/config["alldatafiles"]/"qmof_sginfo1_v13.xlsx"

df_qmofv13=pd.read_excel(qmofv13)
df_dynamic=pd.read_csv(ROOT/config["resultstestdir"]/"dynamic.csv")
df_clamped=pd.read_csv(ROOT/config["resultstestdir"]/"clamped.csv")
df_totale=pd.read_csv(ROOT/config["resultstestdir"]/"totale.csv")


df_checklist=[df_clamped,df_dynamic]
#print (df_clamped)

df_dynamic['info_optnetmagmom']=df_clamped['info_optnetmagmom'].values
df_dynamic['info_piezonetmagmom']=df_clamped['info_piezonetmagmom'].values
df_totale['info_optnetmagmom']=df_clamped['info_optnetmagmom'].values
df_totale['info_piezonetmagmom']=df_clamped['info_piezonetmagmom'].values


list_noncentro_pgs=['1','2','m','222','mm2','3','32','3m','6','4','6mm','4mm','-6','-6m2','-4','-42m','-43m','23','422','622']
list_centro_pgs=['-1','2/m','mmm','4/m','4/mmm','-3','-3m','6/m','6/mmm','m-3','m-3m']

def magmom_check(df_k1):
    df_qmofv13['outputs.pbe.net_magmom']=df_qmofv13['outputs.pbe.net_magmom'].apply(np.around)
    df_k1['info_optnetmagmom']=abs(df_k1['info_optnetmagmom']).apply(np.around)
    df_k1['info_piezonetmagmom']=abs(df_k1['info_piezonetmagmom']).apply(np.around)
    df_k1['info_magmom_matchopt&piezo']=np.where((df_k1['info_optnetmagmom']==df_k1['info_piezonetmagmom']),"equal_bothsteps","notequal_bothsteps")
    for column in df_k1["info_MOFCode"]:
        qmof_magmom=df_qmofv13.loc[df_qmofv13["name"]==column,"outputs.pbe.net_magmom"].iloc[0]
        obtained_optmagmom=df_k1.loc[df_k1["info_MOFCode"]==column,'info_optnetmagmom'].iloc[0]
        if obtained_optmagmom==qmof_magmom:
            df_k1.loc[df_k1["info_MOFCode"]==column,"info_magmom_matchwithqmof"]="equal"
            df_k1.loc[df_k1["info_MOFCode"]==column,"info_qmofmagmom"]=qmof_magmom
        if obtained_optmagmom!=qmof_magmom:
            #print ("matches",column)
            df_k1.loc[df_k1["info_MOFCode"]==column,
                      "info_magmom_matchwithqmof"]="notequal"
            df_k1.loc[df_k1["info_MOFCode"]==column,"info_qmofmagmom"]=qmof_magmom
    return df_k1

def check_filters(csv_names):
    for csv_k,df_k in zip(csv_names,df_checklist):
        print (csv_k)
        df_k=magmom_check(df_k)
        for i in list_noncentro_pgs:
            pg_loc=df_k.index[df_k['info_PointGroup']==i].tolist()
            #print (pg_loc)
            df_updated1= echecks_pointgroup_zeroconv(df_k,i,pg_loc,pow(10,-3))
            df_updated2= echecks_pointgroup_depconv(df_updated1,i,pg_loc,pow(10,-3))
        for j in list_centro_pgs:
            pg_loc1=df_updated2.index[df_updated2['info_PointGroup']==j].tolist()
            for m in pg_loc1:
                df_updated2.at[m,"Status_dependentconv"]="Centro as per csd"
                df_updated2.at[m,"Status_zeroconvergence"]="Centro as per csd"
        df_updated2.to_csv(csv_k)

def mofs_passedfilters(totalecsv):
    df_clamped_passed=pd.read_csv(csv_names[0])
    df_dynamic_passed=pd.read_csv(csv_names[1])
    df_clamped_converged= df_clamped_passed.query('`info_magmom_matchopt&piezo`=="equal_bothsteps" & `info_magmom_matchwithqmof`=="equal" & (`Status_zeroconvergence`=="Success at all eik" or `Status_zeroconvergence`=="This sg doesnt have any")  & (`Status_dependentconv`=="Success at all eik" or `Status_dependentconv`=="This sg doesnt have any")')
    df_dynamic_converged= df_dynamic_passed.query('`info_magmom_matchopt&piezo`=="equal_bothsteps" & `info_magmom_matchwithqmof`=="equal" & (`Status_zeroconvergence`=="Success at all eik" or `Status_zeroconvergence`=="This sg doesnt have any")  & (`Status_dependentconv`=="Success at all eik" or `Status_dependentconv`=="This sg doesnt have any")')
    #print (df_clamped_converged)
    for column1 in df_totale["info_MOFCode"]:
        #print (df_qmofv13.loc[df_qmofv13["name"]==column1,"csd.Structure.Type"].iloc[0])
        df_totale.loc[df_totale["info_MOFCode"] == column1,
                      "info_ClampedNorm"] = df_clamped_passed.loc[df_clamped_passed["info_MOFCode"] == column1, "info_ClampedNorm"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"] == column1,
                      "info_DynamicNorm"] = df_dynamic_passed.loc[df_dynamic_passed["info_MOFCode"] == column1, "info_DynamicNorm"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"]==column1,
                    "info_smilesnodes"]=df_qmofv13.loc[df_qmofv13["name"]==column1,"info.mofid.smiles_nodes"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"]==column1,
                    "info_smileslinkers"]=df_qmofv13.loc[df_qmofv13["name"]==column1,"info.mofid.smiles_linkers"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"] == column1,
                      "info_natoms"] = df_qmofv13.loc[df_qmofv13["name"] == column1, "info.natoms"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"] == column1,
                      "info.density"] = df_qmofv13.loc[df_qmofv13["name"] == column1, "info.density"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"]==column1,
                    "info_Structuretype"]=df_qmofv13.loc[df_qmofv13["name"]==column1,"csd.Structure.Type"].iloc[0]
        df_totale.loc[df_totale["info_MOFCode"]==column1,
                    "info_qmofcsdequal?"]=df_qmofv13.loc[df_qmofv13["name"]==column1,"csd.qmof.equal?"].iloc[0]
        if (column1 in df_clamped_converged.values) and (column1 in df_dynamic_converged.values):
            df_totale.loc[df_totale["info_MOFCode"]==column1,
                        "info_ClampedDynamic_convergence"]="Converged"
        else:
            df_totale.loc[df_totale["info_MOFCode"]==column1,
                        "info_ClampedDynamic_convergence"]="NotConverged"
    df_totale.to_csv(totalecsv)
    return df_totale



In [5]:
"""check_filter function should be done for convergence checks on magmom equality in opt and piezo steps, magmom equality in qmof database and this work, 
convergence of 'eik' to zero and convergence of dependent 'eik' based on the point group. This is checked in Clamped and Dynamic eiks separately."""
csv_names=[ROOT/config["resultstestdir"]/"clamped_edited.csv",ROOT/config["resultstestdir"]/"dynamic_edited.csv"]
check_filters(csv_names)


/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/results_testcase/clamped_edited.csv
/Users/srinidhi/Desktop/chatgpt_analysis/gitpractice/piezoelectric_screening/results_testcase/dynamic_edited.csv


In [6]:
"""The MOFs which satisfy the convergence criteria in dynamic and clamped eik components are then indicated as converged or not converged 
in the totale csv file"""
df_totale_temp=mofs_passedfilters(ROOT/config["resultstestdir"]/"totale_edited.csv")